<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Setting-Up-pPython-Runtime-Environment" data-toc-modified-id="Setting-Up-pPython-Runtime-Environment-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Setting Up pPython Runtime Environment</a></span><ul class="toc-item"><li><span><a href="#Define-the-details-about-the-pPython-run-mode" data-toc-modified-id="Define-the-details-about-the-pPython-run-mode-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Define the details about the pPython run mode</a></span></li></ul></li><li><span><a href="#Writing-pPython-Program:-Hello-World" data-toc-modified-id="Writing-pPython-Program:-Hello-World-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Writing pPython Program: Hello World</a></span><ul class="toc-item"><li><span><a href="#Code:-Hello-World-Version-1" data-toc-modified-id="Code:-Hello-World-Version-1-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Code: Hello World Version 1</a></span></li><li><span><a href="#Run-Hello-World-Version-1" data-toc-modified-id="Run-Hello-World-Version-1-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Run Hello World Version 1</a></span></li><li><span><a href="#Code:-Hello-World-Version-2" data-toc-modified-id="Code:-Hello-World-Version-2-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Code: Hello World Version 2</a></span></li><li><span><a href="#Run-Hello-World-Version-2" data-toc-modified-id="Run-Hello-World-Version-2-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Run Hello World Version 2</a></span></li><li><span><a href="#Code:-Hello-World-Version-3" data-toc-modified-id="Code:-Hello-World-Version-3-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Code: Hello World Version 3</a></span></li><li><span><a href="#Run-Hello-World-Version-3" data-toc-modified-id="Run-Hello-World-Version-3-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Run Hello World Version 3</a></span></li></ul></li></ul></div>

# Setting Up pPython Runtime Environment
<strong>This Jupyter notebook assumes that pPython configuration is already done.</strong>

In [10]:
import platform
import os
import re
import psutil

# Set the LLSC server name for samba/nfs service
LLSC_SERVER = 'txg-gridfs'

# Set GRID_MOUNT_PATH empty
GRID_MOUNT_PATH = ''

system_name = platform.system()
if system_name == 'Windows':
    USER = os.getenv('USERNAME')
else:
    USER = os.getenv('USER')
    
# Check LLGrid Home directory is mapped on your local system
# if not, set up your local macione for LLSC environment

# Extract mount path for LLGrid
system_name = platform.system()
if system_name == 'Windows':
    import win32net
    my_net_drive = win32net.NetUseEnum(None,0)[0]
    for drive in my_net_drive:
        # print('%s'%(drive['remote']))
        if re.search(LLSC_SERVER,drive['remote']):
            GRID_MOUNT_PATH = drive['local']
            break
else:
    USER = os.getenv('USER')
    for sdiskpart in psutil.disk_partitions(all=True):
        # print(sdiskpart)
        # print('%s'%(sdiskpart[0]))
        if re.search(LLSC_SERVER,sdiskpart[0]):
            rint('Mount path: %s'%(sdiskpart[1]))
            GRID_MOUNT_PATH = sdiskpart[1]
            break

# Check variables needed to set PPYTHON_HOME environment variable for local machine 
print('User: %s'%(USER))
print('GRID_MOUNT_PATH: %s'%(GRID_MOUNT_PATH))

if os.path.exists('/etc/llgrid.id'):
    # On a LLSC machine
    HOME = os.getenv('HOME')
    PPYTHON_HOME = HOME+os.sep+'llgrid_beta'+os.sep+'pPython'+os.sep+'latest'
else:
    # On a local machine
    # LLSC HOME directory must be mapped/mounted
    if len(GRID_MOUNT_PATH) > 0 and os.path.exists(GRID_MOUNT_PATH):
        PPYTHON_HOME = GRID_MOUNT_PATH+os.sep+'llgrid_beta'+os.sep+'pPython'+os.sep+'latest'
    else:
        print('ERROR: LLSC HOME directory is not mapped/mounted on the local machine.')
        print('       Finish Section 2 first.')

print('PPYTHON_HOME: %s'%(PPYTHON_HOME))

User: ch21778
GRID_MOUNT_PATH: 
PPYTHON_HOME: /home/gridsan/CH21778/llgrid_beta/pPython/latest


## Define the details about the pPython run mode

In [11]:
import sys

# Define PPYTHON_HOME environment variable
os.environ["PPYTHON_HOME"] = PPYTHON_HOME

# Add Python search path for pPython main function
PPYTHON_PATH = PPYTHON_HOME+os.sep+"src"
sys.path.append(PPYTHON_PATH)
# Import PythonMPI launch funciton
from pRUN import *

# Disable HDF5 file locking (Lustre parallel filesystem on LLSC)
os.environ["HDF5_USE_FILE_LOCKING"]="FALSE"

# By default, local filesystem based messaging kernel is enabled.
# For interactive job, disable it.
os.environ['PPYTHON_LOCAL_FS'] = 'no'

Minimum PythonMPI functions are loaded for pRUN().
gridPython functions are loaded......


# Writing pPython Program: Hello World

## Code: Hello World Version 1
<br><strong>Print "Hello World!"</strong>

In [18]:
src_file_v1 = 'hello_world_v1.py'
print('Read file, %s'%(src_file_v1))
print('-------------')
f = open(src_file_v1, "r")
print(f.read())
f.close()

Read file, hello_world_v1.py
-------------
#
# Just print 'Hello World!'
#
print('Hello World!')



## Run Hello World Version 1

In [19]:
# PythonMPI script filename
py_file = 'hello_world_v1.py'

# Define number of MPI processes
n_proc = 4

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

grid_config_local: PPYTHON_HOME = /home/gridsan/CH21778/llgrid_beta/pPython/latest
--> grid_config_init: updated grid_config with a local configuration, /home/gridsan/CH21778/ppython_conf/grid_config_local.py.
Running: hello_world_v1.py
... scancel 61812849
--> pyMPI_Comm_settings: updated machine_db_settings with a local configuration, /home/gridsan/CH21778/ppython_conf/pyMPI_Comm_settings_local.py.
Launching MPI rank: 3 on grid_slurm_3.
Launching MPI rank: 2 on grid_slurm_2.
Launching MPI rank: 1 on grid_slurm_1.
Launching MPI rank: 0 on c-3-1-3.

Submitted batch job 61812854

grid_run: executing hello_world_v1.py in the current python process.
 
Hello World!


In [20]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

MPI_COMM_WORLD.pkl
PythonMPIdefs1.py
PythonMPIdefs2.py
PythonMPIdefs3.py
Unix_Commands.1.sh
Unix_Commands.2.sh
Unix_Commands.3.sh
Unix_Commands.sh
hello_world_v1.1.out
hello_world_v1.2.out
hello_world_v1.3.out
pRUN.err
pRUN.log
pid.slurm.61812854


In [21]:
#Check the standard output from the other processes
for i in range(3):
    fname = "PythonMPI"+os.sep+"hello_world_v1."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()

File: PythonMPI/hello_world_v1.1.out
-------
Hello World!

File: PythonMPI/hello_world_v1.2.out
-------
Hello World!

File: PythonMPI/hello_world_v1.3.out
-------
Hello World!



## Code: Hello World Version 2
<br>Print "Hello World!"
<br><strong>Print "My process ID (Pid or Rank)"</strong>
<br><strong>Print "How many pPython processes are running (Np)"</strong>

In [22]:
src_file_v2 = 'hello_world_v2.py'
print('Read file, %s'%(src_file_v2))
print('-------------')
f = open(src_file_v2, "r")
print(f.read())
f.close()

Read file, hello_world_v2.py
-------------
import pPython as GPC

# Just print 'Hello World!'
print('Hello World!')

# My pPython process ID (Pid or Rank)
Pid = GPC.Pid
print('My Pid = %d'%(Pid))

# How many pPython processes are running?
Np = GPC.Np
print('Total number of pPython processes (Np) = %d'%(Np))



## Run Hello World Version 2

In [26]:
# PythonMPI script filename
py_file = 'hello_world_v2.py'

# Define number of MPI processes
n_proc = [1, 4, 1]

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

grid_config_local: PPYTHON_HOME = /home/gridsan/CH21778/llgrid_beta/pPython/latest
--> grid_config_init: updated grid_config with a local configuration, /home/gridsan/CH21778/ppython_conf/grid_config_local.py.


Your triples mode job [1,4,1] requests 4*1 = 4 < 48 threads on the node.
You may not be using all of the 48 cores on the node you requested, which may be sub-optimal.
See https://supercloud.mit.edu/faq#triples-warning for more information.


Running: hello_world_v2.py
... scancel 61812858
--> pyMPI_Comm_settings: updated machine_db_settings with a local configuration, /home/gridsan/CH21778/ppython_conf/pyMPI_Comm_settings_local.py.
Launching MPI rank: 3 to 1 on grid_slurm_1.
Launching MPI rank: 0 to 0 on c-3-1-3.

Submitted batch job 61812872

grid_run: executing hello_world_v2.py in the current python process.
 
Hello World!
My Pid = 0
Total number of pPython processes (Np) = 4


In [27]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

MPI_COMM_WORLD.pkl
PythonMPIdefs.py
Unix_Commands.1.sh
Unix_Commands.sh
p1-p3_c-15-10-3
pRUN.err
pRUN.log
pid.slurm.61812872


In [28]:
#Check the standard output from the other processes
my_list = os.listdir('PythonMPI')
for entry in my_list:
    if re.search('p1-',entry):
        #Find the subdirectory for the first compute node 
        subdir_name = entry
        break
for i in range(3):
    fname = "PythonMPI"+os.sep+subdir_name+os.sep+"hello_world_v2."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()

File: PythonMPI/p1-p3_c-15-10-3/hello_world_v2.1.out
-------
Hello World!
My Pid = 1
Total number of pPython processes (Np) = 4

File: PythonMPI/p1-p3_c-15-10-3/hello_world_v2.2.out
-------
Hello World!
My Pid = 2
Total number of pPython processes (Np) = 4

File: PythonMPI/p1-p3_c-15-10-3/hello_world_v2.3.out
-------
Hello World!
My Pid = 3
Total number of pPython processes (Np) = 4



## Code: Hello World Version 3
<br>Print "Hello World!"
<br>Print "My process ID (Pid or Rank)"
<br>Print "How many pPython processes are running (Np)"
<br><strong>Create a distributed random array</strong>
<br><strong>Do some mathematical operations in parallel</strong>
<br><strong>Aggregate the result and display on the leader process</strong>

In [33]:
src_file_v2 = 'hello_world_v3.py'
print('Read file, %s'%(src_file_v2))
print('-------------')
f = open(src_file_v2, "r")
print(f.read())
f.close()

Read file, hello_world_v3.py
-------------
import pPython as GPC
from Dmap import *
from rand import *
from local import *
from global_ind import *
from put_local import *
from agg import *

# Just print 'Hello World!'
print('Hello World!')

# My pPython process ID (Pid or Rank)
Pid = GPC.Pid
print('My Pid = %d'%(Pid))

# How many pPython processes are running?
Np = GPC.Np
print('Total number of pPython processes (Np) = %d'%(Np))

# Create a distributed array
Nx = 4
Ny = 12
amap  = Dmap([1,Np], {}, range(Np))
Z = rand(Nx,Ny,map=amap)
myZ = local(Z)
print('My local portion of the distributed array, Z:')
print(myZ)

# Do some mathematical operation in parallel
# Get the local portion of the global indices
my_index = global_ind(Z, 1)
# Loop over the local indices
for index in range(len(my_index)):
    for j in range(Nx):
        myZ[j,index] = myZ[j,index] + 0.

# Store the local portion (myZ) of Z into the distributed matrix Z
Z = put_local(Z, myZ)

# Finally, aggregate all of the output

## Run Hello World Version 3

In [34]:
# PythonMPI script filename
py_file = 'hello_world_v3.py'

# Define number of MPI processes
n_proc = [1, 4, 1]

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

grid_config_local: PPYTHON_HOME = /home/gridsan/CH21778/llgrid_beta/pPython/latest
--> grid_config_init: updated grid_config with a local configuration, /home/gridsan/CH21778/ppython_conf/grid_config_local.py.


Your triples mode job [1,4,1] requests 4*1 = 4 < 48 threads on the node.
You may not be using all of the 48 cores on the node you requested, which may be sub-optimal.
See https://supercloud.mit.edu/faq#triples-warning for more information.


Running: hello_world_v3.py
... scancel 61813430
--> pyMPI_Comm_settings: updated machine_db_settings with a local configuration, /home/gridsan/CH21778/ppython_conf/pyMPI_Comm_settings_local.py.
Launching MPI rank: 3 to 1 on grid_slurm_1.
Launching MPI rank: 0 to 0 on c-3-1-3.

Submitted batch job 61813486

grid_run: executing hello_world_v3.py in the current python process.
 
Hello World!
My Pid = 0
Total number of pPython processes (Np) = 4
My local portion of the distributed array, Z:
[[0.74806958 0.35063153 0.23998034]
 [0.39077347 0.322

In [35]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

MPI_COMM_WORLD.pkl
PythonMPIdefs.py
Unix_Commands.1.sh
Unix_Commands.sh
p1-p3_c-15-11-1
pRUN.err
pRUN.log
pid.slurm.61813486


In [37]:
#Check the standard output from the other processes
my_list = os.listdir('PythonMPI')
for entry in my_list:
    if re.search('p1-',entry):
        #Find the subdirectory for the first compute node 
        subdir_name = entry
        break
for i in range(3):
    fname = "PythonMPI"+os.sep+subdir_name+os.sep+"hello_world_v3."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()

File: PythonMPI/p1-p3_c-15-11-1/hello_world_v3.1.out
-------
Hello World!
My Pid = 1
Total number of pPython processes (Np) = 4
My local portion of the distributed array, Z:
[[0.53897047 0.89064263 0.68731307]
 [0.75928329 0.46847514 0.58814162]
 [0.81352772 0.20947064 0.50902798]
 [0.81034724 0.59730597 0.57726849]]
Final z_final on Pid = 1:
[[0.53897047 0.89064263 0.68731307]
 [0.75928329 0.46847514 0.58814162]
 [0.81352772 0.20947064 0.50902798]
 [0.81034724 0.59730597 0.57726849]]

File: PythonMPI/p1-p3_c-15-11-1/hello_world_v3.2.out
-------
Hello World!
My Pid = 2
Total number of pPython processes (Np) = 4
My local portion of the distributed array, Z:
[[0.04767775 0.394957   0.13736869]
 [0.38060598 0.3995587  0.86210421]
 [0.68401224 0.94985758 0.7055211 ]
 [0.00113903 0.68761147 0.57466921]]
Final z_final on Pid = 2:
[[0.04767775 0.394957   0.13736869]
 [0.38060598 0.3995587  0.86210421]
 [0.68401224 0.94985758 0.7055211 ]
 [0.00113903 0.68761147 0.57466921]]

File: PythonMPI/p1